# Corporate Reorganization — ModernBERT retriever training (SageMaker)

This notebook launches a SageMaker training job using the code in `corporate_reorganization/modernbert/`.

Prereq: build the processed dataset first (so `../data/final_annotations_gold/processed/` exists).

In [2]:

import os, sys
print("Has var:", "SAGEMAKER_EXECUTION_ROLE_ARN" in os.environ)
print("CWD:", os.getcwd())
print("Python:", sys.executable)
print(os.getenv("SAGEMAKER_EXECUTION_ROLE_ARN"))
print("SAGEMAKER_EXECUTION_ROLE_ARN" in os.environ)
print(os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN"))



Has var: False
CWD: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/notebooks
Python: /home/lbrenap/miniconda3/envs/legalpacaenv/bin/python
None
False
None


In [5]:
from pathlib import Path

import sagemaker
from sagemaker.huggingface import HuggingFace
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

# set aws variables
print(os.environ["SAGEMAKER_EXECUTION_ROLE_ARN"])
role = os.environ["SAGEMAKER_EXECUTION_ROLE_ARN"] #from environment variable
session = sagemaker.Session()
bucket = session.default_bucket()
 # default
prefix = "corporate_reorganization/retriever"

arn:aws:iam::371087393859:role/defaultrole


In [23]:
processed_dir = Path("../data/final_annotations_gold/processed").resolve()
assert processed_dir.exists(), f"Missing processed_dir: {processed_dir}"

data_s3_uri = session.upload_data(
    path=str(processed_dir),
    bucket=bucket,
    key_prefix=f"{prefix}/data/processed",
)

inputs = {"data": data_s3_uri}
data_s3_uri

's3://sagemaker-us-east-1-371087393859/corporate_reorganization/retriever/data/processed'

In [24]:
metric_definitions = [
    {"Name": "eval_loss", "Regex": r"SM_METRIC eval_loss=([0-9eE\.\-]+)"},
    {"Name": "eval_set_recall_at_20", "Regex": r"SM_METRIC eval_set_recall_at_20=([0-9eE\.\-]+)"},
    {"Name": "eval_mrr", "Regex": r"SM_METRIC eval_mrr=([0-9eE\.\-]+)"},
    {"Name": "eval_retrieval_loss", "Regex": r"SM_METRIC eval_retrieval_loss=([0-9eE\.\-]+)"},
    {"Name": "eval_avg_candidates", "Regex": r"SM_METRIC eval_avg_candidates=([0-9eE\.\-]+)"},
]

metric_definitions

[{'Name': 'eval_loss', 'Regex': 'SM_METRIC eval_loss=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_set_recall_at_20',
  'Regex': 'SM_METRIC eval_set_recall_at_20=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_mrr', 'Regex': 'SM_METRIC eval_mrr=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_retrieval_loss',
  'Regex': 'SM_METRIC eval_retrieval_loss=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_avg_candidates',
  'Regex': 'SM_METRIC eval_avg_candidates=([0-9eE\\.\\-]+)'}]

In [ ]:
hyperparameters = {
    "deepspeed": "ds_zero3.json",
    "epochs": 20,
    "learning_rate": 3e-5,
    "temperature": 0.05,
    "batch_size_queries": 8,
    "max_len_query": 4096,
    "max_len_passage": 600,
    "max_pos_per_query": 4,
    "num_same_case_negatives": 48,
    "num_distractor_negatives": 4,
    "distractor_labels": "Background Facts",
    "eval_query_batch_size": 64,
    "eval_passage_batch_size": 256,
    "gradient_accumulation_steps":4,
}

estimator = HuggingFace(
    entry_point="train_sm.py",
    source_dir="../modernbert",
    instance_type="ml.g5.12xlarge",
    instance_count=1,
    role=role,
    transformers_version="4.49.0",
    pytorch_version="2.5.1",
    py_version="py311",
    hyperparameters=hyperparameters,
    metric_definitions=metric_definitions,
    distribution={"mpi": {"enabled": True, "processes_per_host": 4}},
)

estimator.fit(inputs)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-01-05-07-23-40-734


2026-01-05 07:23:42 Starting - Starting the training job
2026-01-05 07:23:42 Pending - Training job waiting for capacity...............
2026-01-05 07:26:05 Pending - Preparing the instances for training...
2026-01-05 07:26:47 Downloading - Downloading the training image...........

## Output artifacts

After the job finishes, the model artifact is in `estimator.model_data` (an S3 `model.tar.gz`).

Inside `model.tar.gz` (under `/opt/ml/model` during training), the script writes:

- `model.safetensors`
- `wrapper_config.json`
- tokenizer files (`tokenizer.json`, `special_tokens_map.json`, etc.)
- `encoder_config/` (base encoder config)